# Tutorial 05: Detección de Estrés por Actividad Electro dérmica (EDA/GSR)
### TinyML para Raspberry Pi Pico 2 / Pico 2 W (RP2350) con TensorFlow INT8

En este tutorial aprenderás:
1. **Fisiología de la Actividad Electrodérmica (EDA/GSR)**: Nivel Tónico (SCL - Skin Conductance Level) y Respuestas Fásicas (SCR - Skin Conductance Responses) vinculadas al sistema nervioso autónomo simpático.
2. **Generación con NeuroKit2** de perfiles basales en calma vs estados de alta activación / estrés psicofisiológico.
3. **Red Convolucional 1D Ultra-ligera para Series de Tiempo**.
4. **Cuantización INT8 con calibración** para ejecución en microsegundos en el procesador ARM Cortex-M33 a 150 MHz.
5. **Despliegue a Arduino IDE (`model_data.h`) para Raspberry Pi Pico 2**.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.signal_generator import generate_eda_dataset
from utils.exporter import tflite_to_c_header, save_test_samples_header
from utils.tflite_eval import evaluate_tflite_model

## 1. Generación de Señales EDA con NeuroKit2

In [ ]:
SAMPLE_LEN = 200
SAMPLING_RATE = 50
CLASS_NAMES = ["Calm_Baseline", "Stress_Arousal"]

X, y = generate_eda_dataset(num_samples_per_class=400, sample_len=SAMPLE_LEN, sampling_rate=SAMPLING_RATE)
X = np.expand_dims(X, axis=-1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
t_axis = np.arange(SAMPLE_LEN) / SAMPLING_RATE
for c in range(2):
    idx = np.where(y_train == c)[0][0]
    axes[c].plot(t_axis, X_train[idx, :, 0], color="royalblue" if c==0 else "crimson")
    axes[c].set_title(f"{CLASS_NAMES[c]}", fontweight='bold')
    axes[c].set_xlabel("Tiempo (s)")
    axes[c].grid(True, alpha=0.5)
axes[0].set_ylabel("Conductancia (uS) Normalizada")
plt.suptitle("Bioseñales EDA: Reposo vs Estimulación Simpática")
plt.tight_layout()
plt.show()

## 2. Construcción y Entrenamiento del Modelo 1D-CNN

In [ ]:
def build_eda_cnn():
    inputs = tf.keras.Input(shape=(SAMPLE_LEN, 1))
    x = tf.keras.layers.Conv1D(8, 7, strides=2, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPool1D(2)(x)
    x = tf.keras.layers.Conv1D(16, 5, strides=2, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(16, activation="relu")(x)
    outputs = tf.keras.layers.Dense(2, activation="softmax")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

model = build_eda_cnn()
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=15, batch_size=32, verbose=1)

_, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Exactitud Modelo EDA: {acc * 100:.2f}%")

## 3. Cuantización a INT8 y Exportación a C para Raspberry Pi Pico 2

In [ ]:
def rep_dataset():
    for i in range(150):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_eda_int8 = converter.convert()

ARDUINO_DIR = Path("../arduino/eda_inference_pico2")
ARDUINO_DIR.mkdir(parents=True, exist_ok=True)

tflite_to_c_header(tflite_eda_int8, str(ARDUINO_DIR / "model_data.h"))
save_test_samples_header(X_test[:, :, 0], y_test, str(ARDUINO_DIR / "test_samples.h"), CLASS_NAMES, samples_per_class=3)
print(f"Modelo EDA exportado ({len(tflite_eda_int8)} bytes). Listo para Arduino!")